# Main Paper Figures

Reproduces Figures 1–4 of Rappeport & Nitzan, *Fitness and Overfitness: Implicit Regularization in Evolutionary Dynamics*.

Most cells **load** pre-computed simulation outputs from `../data/`. To regenerate the cached data, run the matching script in `../experiments/`, e.g.:

```bash
python -m experiments.main_fig2_stable_env
```

Single-realization Muller plots are generated live (fast).

Figure panels 1A, 1B, 1D, 1E are conceptual schematics not produced from simulation; they live in `../figures/main/schematic_*.svg` (or are inserted at typesetting time).

## 0. Setup

In [ ]:
import os, sys, pickle, numpy as np, matplotlib.pyplot as plt
sys.path.insert(0, os.path.abspath('../src'))
from overfitness_paper import (
    run, np_temp_seed, setup_plot_style,
    plot_muller, plot_bubble_chart, plot_dynamics_panel,
    plot_fitness_distribution, plot_env_change_heatmap,
)
from overfitness_paper.config import (
    FITNESS_GAMMA, SIGMA, T, CLASS_SIZE, N, KS, DEMO_SEED,
)
DATA = os.path.abspath('../data')
FIG = os.path.abspath('../figures/main'); os.makedirs(FIG, exist_ok=True)
setup_plot_style()

## Figure 1: Model setup and Muller plot demonstration

Panels A, B, D, E are conceptual schematics (drawn separately).
Panel C is a 3-class Muller plot illustrating the replicator update.
Panel F is a Muller plot of a typical realization at q*=5.

In [ ]:
# Fig 1F — Muller plot of class frequencies (q*=5, T=200, single realization)
with np_temp_seed(DEMO_SEED):
    log = run(exp_name='fig1F', true_k=5, fitness_gamma=FITNESS_GAMMA,
              n=N, T=200, class_size=CLASS_SIZE, xi=SIGMA, ks=KS, to_save=None)
fig, ax = plt.subplots(figsize=(12, 5))
plot_muller(log['class_frequency'].T, ax=ax)
ax.set_xlabel('Generations'); ax.set_ylabel('Frequency')
fig.savefig(f'{FIG}/fig1F_muller.pdf', bbox_inches='tight'); plt.show()

## Figure 2: Selected complexity vs environmental complexity

- **2A**: bubble chart — class containing organism with maximum time-averaged fitness.
- **2B**: bubble chart — selected class (highest time-averaged frequency).
- **2C-F**: dynamics for q*=5 — max fitness per class, fitness of globally optimal member, Occam factor, class growth rate.

In [ ]:
d = np.load(f'{DATA}/fig2_bubbles.npz')
ks = d['ks']
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
plot_bubble_chart(d['bubble_max_fitness_class'], ax=axes[0],
                  xlabel='$q^*$', ylabel='Class with max fitness')
plot_bubble_chart(d['bubble_selected_class'], ax=axes[1],
                  xlabel='$q^*$', ylabel='Selected class')
fig.savefig(f'{FIG}/fig2AB_bubbles.pdf', bbox_inches='tight'); plt.show()

In [ ]:
d = np.load(f'{DATA}/fig2_dynamics_q5.npz')
ks = d['ks']
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for ax, key, ylabel in [
    (axes[0, 0], 'max_fitness', 'Max fitness per class'),
    (axes[0, 1], 'optimal_member_fitness', 'Fitness of optimal member'),
    (axes[1, 0], 'occam_factor', 'Occam factor'),
    (axes[1, 1], 'class_growth_rate', 'Class growth rate'),
]:
    plot_dynamics_panel(d[key], ks, ax=ax, ylabel=ylabel)
fig.savefig(f'{FIG}/fig2CF_dynamics.pdf', bbox_inches='tight'); plt.show()

## Figure 3: Transient success of simple classes

- **3A**: Muller of a typical realization at q*=5.
- **3B**: per-class fitness distributions (3D surface).
- **3C-F**: time series of class fitness, max fitness, Occam factor, growth rate.

In [ ]:
# 3A — single Muller realization
with np_temp_seed(DEMO_SEED + 3):
    log = run(exp_name='fig3A', true_k=5, fitness_gamma=FITNESS_GAMMA,
              n=N, T=100, class_size=CLASS_SIZE, xi=SIGMA, ks=KS, to_save=None)
fig, ax = plt.subplots(figsize=(10, 5))
plot_muller(log['class_frequency'].T, ax=ax)
ax.set_xlabel('Generations'); ax.set_ylabel('Frequency')
fig.savefig(f'{FIG}/fig3A_muller.pdf', bbox_inches='tight'); plt.show()

In [ ]:
# 3B — per-class fitness distribution surface
d = np.load(f'{DATA}/fig3_fitness_dist.npz', allow_pickle=True)
ks = d['ks']
fitness_by_class = [d[f'class_{i}'] if f'class_{i}' in d.files else d['class_fitness_t0'][i]
                    for i in range(len(ks))]
fig = plt.figure(figsize=(8, 5))
ax = fig.add_subplot(111, projection='3d')
plot_fitness_distribution(fitness_by_class, ks, ax=ax)
fig.savefig(f'{FIG}/fig3B_fitness_dist.pdf', bbox_inches='tight'); plt.show()

In [ ]:
# 3C-F — time series
d = np.load(f'{DATA}/fig3_dynamics.npz')
ks = d['ks']
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for ax, key, ylabel in [
    (axes[0, 0], 'class_fitness', 'Class fitness'),
    (axes[0, 1], 'max_fitness', 'Max fitness'),
    (axes[1, 0], 'occam_factor', 'Occam factor'),
    (axes[1, 1], 'class_growth_rate', 'Class growth rate'),
]:
    mean = d[key].mean(axis=0)  # average over realizations -> (T, n_ks)
    for j, k in enumerate(ks):
        ax.plot(mean[:, j], label=f'q={k}')
    ax.set_xlabel('Generations'); ax.set_ylabel(ylabel)
axes[0, 0].legend(fontsize=7, loc='best', ncol=2)
fig.savefig(f'{FIG}/fig3CF_dynamics.pdf', bbox_inches='tight'); plt.show()

## Figure 4: Environmental change selects for reduced complexity

- **4A**: Muller — env changes once mid-simulation.
- **4B**: Muller — env changes rapidly.
- **4C-D**: time series under rapid env change.
- **4E**: heatmap of selected complexity vs (q*, env change rate).

In [ ]:
with open(f'{DATA}/fig4_mullers.pkl', 'rb') as f:
    mullers = pickle.load(f)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, label in zip(axes, ['single_change', 'rapid_change']):
    plot_muller(mullers[label]['class_frequency'].T, ax=ax)
    ax.set_xlabel('Generations'); ax.set_ylabel('Frequency')
    ax.set_title(label)
fig.savefig(f'{FIG}/fig4AB_mullers.pdf', bbox_inches='tight'); plt.show()

In [ ]:
d = np.load(f'{DATA}/fig4_dynamics.npz')
ks = d['ks']
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, key, ylabel in [
    (axes[0], 'max_fitness_per_class', 'Max fitness'),
    (axes[1], 'occam_factor', 'Occam factor'),
    (axes[2], 'class_growth_rate', 'Class growth rate'),
]:
    mean = d[key].mean(axis=0)
    for j, k in enumerate(ks):
        ax.plot(mean[:, j], label=f'q={k}')
    ax.set_xlabel('Generations'); ax.set_ylabel(ylabel)
axes[0].legend(fontsize=7, ncol=2)
fig.savefig(f'{FIG}/fig4CD_dynamics.pdf', bbox_inches='tight'); plt.show()

In [ ]:
d = np.load(f'{DATA}/fig4e_heatmap.npz')
fig, ax = plt.subplots(figsize=(7, 5))
plot_env_change_heatmap(d['mean_selected_class'], d['ks'], d['n_envs_list'], ax=ax)
fig.savefig(f'{FIG}/fig4E_heatmap.pdf', bbox_inches='tight'); plt.show()